In [1]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer
from sklearn.utils import check_random_state

#from aeon.transformations.base import BaseCollectionTransformer, BaseTransformer
from aeon.transformations.base import BaseTransformer
from aeon.transformations.collection import BaseCollectionTransformer, Normalizer
from aeon.transformations.collection import PeriodogramTransformer
from aeon.transformations.collection.interval_based import RandomIntervals
from aeon.transformations.collection.feature_based import Catch22
from aeon.utils.numba.general import first_order_differences_3d
from aeon.utils.numba.stats import (
    row_mean,
    row_std,
    row_slope,
    row_median,
    row_iqr,
    row_numba_min,
    row_numba_max,
)
from aeon.utils.validation import check_n_jobs

In [2]:
class DrCIFFeatureExtractor(BaseCollectionTransformer):
    """
    Feature-only extractor using the same default representations and
    interval features as DrCIF.

    No trees, no estimators.
    """

    _tags = {
        "X_inner_type": ["numpy3D"],
        "capability:multivariate": True,
        "fit_is_empty": False,
    }

    def __init__(
        self,
        series_transformers=None,
        interval_features=None,
        n_intervals=(4, "sqrt-div"),
        min_interval_length=3,
        max_interval_length=0.5,
        use_pycatch22=False,
        random_state=None,
        n_jobs=1,
    ):
        # ---- DrCIF defaults
        if series_transformers is None:
            series_transformers = [
                None,
                FunctionTransformer(
                    func=first_order_differences_3d, validate=False
                ),
                PeriodogramTransformer(),
            ]

        if interval_features is None:
            interval_features = [
                Catch22(
                    outlier_norm=True,
                    use_pycatch22=use_pycatch22,
                ),
                row_mean,
                row_std,
                row_slope,
                row_median,
                row_iqr,
                row_numba_min,
                row_numba_max,
            ]

        self.series_transformers = series_transformers
        self.interval_features = interval_features
        self.n_intervals = n_intervals
        self.min_interval_length = min_interval_length
        self.max_interval_length = max_interval_length
        self.use_pycatch22 = use_pycatch22
        self.random_state = random_state
        self.n_jobs = n_jobs
        super().__init__()

        if use_pycatch22:
            self.set_tags(**{"python_dependencies": "pycatch22"})

    # --------------------------------------------------
    # FIT
    # --------------------------------------------------
    def _fit(self, X, y=None):
        rng = check_random_state(self.random_state)
        self.n_cases_, self.n_channels_, self.n_timepoints_ = X.shape
        self._n_jobs = check_n_jobs(self.n_jobs)

        # ---- apply series transformers
        Xt = []
        self._series_transformers_ = []

        for tr in self.series_transformers:
            if tr is None:
                Xt.append(X)
                self._series_transformers_.append(None)
            elif isinstance(tr, (BaseTransformer, FunctionTransformer)):
                t = tr
                Xt.append(t.fit_transform(X, y))
                self._series_transformers_.append(t)
            else:
                raise ValueError(f"Invalid transformer: {tr}")

        # ---- resolve number of intervals (same logic as base class)
        self._n_intervals_ = []
        for series in Xt:
            t = series.shape[2]
            total = 0
            for v in self.n_intervals:
                if isinstance(v, int):
                    total += v
                elif v == "sqrt":
                    total += int(np.sqrt(t))
                elif v == "sqrt-div":
                    total += int(np.sqrt(t) / len(Xt))
                else:
                    raise ValueError(f"Invalid n_intervals entry: {v}")
            self._n_intervals_.append(max(1, total))

        # ---- resolve interval lengths
        self._min_len_ = []
        self._max_len_ = []

        for series in Xt:
            t = series.shape[2]

            min_len = (
                int(self.min_interval_length * t)
                if isinstance(self.min_interval_length, float)
                else self.min_interval_length
            )
            max_len = (
                int(self.max_interval_length * t)
                if isinstance(self.max_interval_length, float)
                else self.max_interval_length
            )

            min_len = max(3, min(min_len, t))
            max_len = max(min_len, min(max_len, t))

            self._min_len_.append(min_len)
            self._max_len_.append(max_len)

        # ---- build interval selectors
        self._interval_selectors_ = []

        for r in range(len(Xt)):
            selector = RandomIntervals(
                n_intervals=self._n_intervals_[r],
                min_interval_length=self._min_len_[r],
                max_interval_length=self._max_len_[r],
                features=self.interval_features,
                random_state=rng.randint(0, 2**31 - 1),
            )
            selector.fit(Xt[r], y)
            self._interval_selectors_.append(selector)

        return self

    # --------------------------------------------------
    # TRANSFORM
    # --------------------------------------------------
    def _transform(self, X, y=None):
        Xt = []
        for tr in self._series_transformers_:
            Xt.append(X if tr is None else tr.transform(X))

        features = np.empty((X.shape[0], 0))
        for r, selector in enumerate(self._interval_selectors_):
            f = selector.transform(Xt[r])
            features = np.hstack((features, f))

        return features


In [3]:
from aeon.classification.hybrid import HIVECOTEV2
from joblib import parallel_backend
from autotsc import utils


dataset = "Strawberry"
X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(613, 1, 235) (613,) (370, 1, 235) (370,)


In [4]:
Xt_train = []
Xt_test = []
for i in range(20):
    t = DrCIFFeatureExtractor(n_jobs=-1, random_state=i)
    v1 = t.fit_transform(X_train)
    v2 = t.transform(X_test)
    Xt_train.append(v1)
    Xt_test.append(v2)

Xt_train = np.hstack(Xt_train)
Xt_test = np.hstack(Xt_test)

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


m = RandomForestClassifier(n_jobs=-1, n_estimators=400)
m.fit(Xt_train, y_train)
y_pred = m.predict(Xt_test)
acc = accuracy_score(y_test, y_pred)
print(f"DrCIF features + RF accuracy: {acc:.4f}")

DrCIF features + RF accuracy: 0.9649


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score


m = ExtraTreesClassifier(n_jobs=-1, n_estimators=400)
m.fit(Xt_train, y_train)
y_pred = m.predict(Xt_test)
acc = accuracy_score(y_test, y_pred)
print(f"DrCIF features + RF accuracy: {acc:.4f}")

DrCIF features + RF accuracy: 0.9622


In [ ]:
from aeon.classification.interval_based import DrCIFClassifier
m = DrCIFClassifier(n_jobs=-1, random_state=42, n_estimators=50)
m.fit(X_train, y_train)
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"DrCIF accuracy: {acc:.4f}")

DrCIF accuracy: 0.9703


In [9]:
from aeon.classification.interval_based import DrCIFClassifier
m = DrCIFClassifier(n_jobs=-1, random_state=42, n_estimators=20)
m.__unit_test_flag = True
m.fit(X_train, y_train)
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"DrCIF accuracy: {acc:.4f}")
features_per_tree = m._transformed_data

DrCIF accuracy: 0.9703


In [12]:
features_per_tree[0].shape

(613, 250)

In [15]:
features_per_tree[7].shape

(613, 250)